# Week 2 — Full station inventory + HPC scratch layout
Scope: 1 Aug – 30 Sep 2019, GeoNet (`NZ`) + BARNZ (`YA`) networks.

This checks which FDSN endpoint actually serves BARNZ data (GeoNet's own,
or IRIS), downloads full station+response inventories for both, and sets up
the scratch folder structure the rest of the project will write into.

In [ ]:
import obspy
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
import os

PROJECT_DIR = os.path.expanduser("~/tvz_project")
INVENTORY_DIR = os.path.join(PROJECT_DIR, "data", "inventory")
os.makedirs(INVENTORY_DIR, exist_ok=True)

t0 = UTCDateTime("2019-08-01T00:00:00")
t1 = UTCDateTime("2019-09-30T23:59:59")

# Same ~1 deg radius around Lake Taupo used in the Week 1 toy test
LAT, LON, RADIUS = -38.8, 176.0, 1.0

In [ ]:
# GeoNet permanent network (NZ) — this one we already know works via GEONET client
geonet_client = Client("GEONET")

nz_inventory = geonet_client.get_stations(
    network="NZ", station="*",
    latitude=LAT, longitude=LON, maxradius=RADIUS,
    starttime=t0, endtime=t1,
    level="response",   # full response info — needed later for magnitude estimation
)
print(nz_inventory)
nz_inventory.write(os.path.join(INVENTORY_DIR, "NZ_2019-08-01_2019-09-30.xml"), format="STATIONXML")

In [ ]:
# BARNZ (network code YA, from the Ebinger & Stern 2017 dataset DOI: 10.7914/SN/YA_2017).
# Not all temporary IRIS-registered networks are mirrored on GeoNet's own FDSN endpoint,
# so try GEONET first and fall back to IRIS if that comes back empty.
ya_inventory = None

for client_name in ["GEONET", "IRIS"]:
    try:
        client = Client(client_name)
        candidate = client.get_stations(
            network="YA", station="*",
            starttime=t0, endtime=t1,
            level="response",
        )
        n_stations = sum(len(net) for net in candidate)
        print(f"{client_name}: found {n_stations} YA stations")
        if n_stations > 0:
            ya_inventory = candidate
            print(f"-> Using {client_name} as the BARNZ (YA) data source going forward")
            break
    except Exception as e:
        print(f"{client_name}: query failed — {e}")

if ya_inventory is not None:
    print(ya_inventory)
    ya_inventory.write(os.path.join(INVENTORY_DIR, "YA_2019-08-01_2019-09-30.xml"), format="STATIONXML")
else:
    print("WARNING: BARNZ (YA) not found on either endpoint for this date range.")
    print("Possible causes: array had already been decommissioned by Aug 2019 (check exact")
    print("end date), or it needs a different network code / another data centre entirely.")

In [ ]:
# Scratch folder structure for the rest of the project
subfolders = [
    "data/waveforms",
    "data/inventory",
    "picks/eqtransformer",
    "picks/phasenet",
    "catalogs/eqtransformer",
    "catalogs/phasenet",
    "catalogs/quakemigrate",
    "velocity_models",
    "logs",
]
for sub in subfolders:
    path = os.path.join(PROJECT_DIR, sub)
    os.makedirs(path, exist_ok=True)
    print(f"Created: {path}")